In [1]:
import pandas as pd
from pathlib import Path
import os
import ast

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# Define in and out destinations

root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch" + "/"
#root = "Y:/_Tobias/LSM980/Mitotic_Stopwatch" + "/"
outputFolder = root + "DataFrames" + "/"

In [ ]:
# Define time interval between frames in minutes
frame_interval = 7

In [3]:
df = pd.read_csv(root + '/DataFrames/MainDataFrame_Lineages_Augmin.csv')

Index(['Unnamed: 0', 'Track_ID', 'Source_ID', 'Splitting_event', 'Lineage',
       'Track_Coordinate_X', 'Track_Coordinate_Y', 'Frame', 'EndPoint_ID',
       'Position', 'Dataset', 'Condition', 'Generation', 'Mother_ID',
       'Grandmother_ID', 'Sister_ID', 'Daughters_ID',
       'Number_of_daughters_in_mitosis',
       'Mother_Number_of_daughters_in_mitosis', 'Cell_Cycle_mins',
       'Cell_Cycle_hours', 'Seen_sister', 'Seen_granny'],
      dtype='object')

In [4]:
# V2 error parsing and conversion


def concatenate_error_analysis_files(root_folder = root):

    root_folder = Path(root_folder)
    
    # Recursively find all matching files
    csv_files = list(root_folder.rglob("*_ErrorAnalysis.csv"))
    
    if not csv_files:
        print("No matching files found.")
        return pd.DataFrame()
    
    # Read and concatenate
    df_list = []
    for file in csv_files:
        try:
            df = pd.read_csv(file)
            df_list.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")
    
    if df_list:
        df_errors_V2 = pd.concat(df_list, ignore_index = True)
    else:
        df_errors_V2 = pd.DataFrame()
    
    return df_errors_V2

def add_error_type_flags(df_errors_V2):
    error_types = [
       # "normal",
       # "blank",
        "Laggings",
        "Misaligned",
        "Cytokinesis Failure",
        "Massive Missegregation",
        "DNA bridge",
        "Multipolar Division",
        "Other",
        "Mitotic Death",
        "Slippage",
        "MN"
    ]
    
    for error in error_types:
        df_errors_V2[error] = (
            (df_errors_V2["1st Error Type"] == error) |
            (df_errors_V2["2nd Error Type"] == error)
        )
    
    # Create "any_true" column
    df_errors_V2["any_true"] = df_errors_V2[error_types].any(axis = 1)
    
    return df_errors_V2

df_errors_V2 = concatenate_error_analysis_files()
df_errors_V2 = add_error_type_flags(df_errors_V2)

df_errors_V2["Unique_Source_ID"] = df_errors_V2.Dataset.astype(str) + df_errors_V2.Position.astype(str) + df_errors_V2.Source_ID.astype(str)

4266
[20250627 20250704 20251204 20251215 20260115 20260126 20260205 20260306]


,Unnamed: 0,Track_ID,Source_ID,Splitting_event,Lineage,Track_Coordinate_X,Track_Coordinate_Y,Frame,EndPoint_ID,Position,...,Cytokinesis Failure,Massive Missegregation,DNA bridge,Multipolar Division,Other,Mitotic Death,Slippage,MN,any_true,Unique_Source_ID
0,17,5,2547,True,2547,344.508,446.431,0,[],3,...,False,False,False,False,True,False,False,False,True,2025062732547
1,20,6,2574,True,2574,349.894,317.163,0,[],3,...,False,False,False,False,False,False,False,False,False,2025062732574
2,35,39,3033,True,3033,243.414,302.247,58,[],3,...,False,False,False,False,False,False,False,False,False,2025062733033
3,68,27,5842,True,5842,603.873,329.178,58,[],3,...,False,False,False,False,False,False,False,False,False,2025062735842
4,67,26,5802,True,5802,766.702,102.545,59,[],3,...,False,False,False,False,False,False,False,False,False,2025062735802
5,4,2,2245,True,2245,380.416,449.193,60,[],3,...,False,False,False,False,False,False,False,False,False,2025062732245
6,71,28,5905,True,5905,462.590,94.672,60,[],3,...,False,False,False,False,False,False,False,False,False,2025062735905
7,65,25,5723,True,5723,509.823,105.445,60,[],3,...,False,False,False,False,False,False,False,False,False,2025062735723
8,52,21,5421,True,5421,571.556,446.017,60,[],3,...,False,False,False,False,False,False,False,False,False,2025062735421
9,12,4,2445,True,2445,582.191,400.442,60,[],3,...,False,False,False,False,False,False,False,False,False,2025062732445


In [5]:
# Parse mitotic error tables (after FIJI analysis)
# Note: these are the first generation datasets only (see the error V2 analysis parsing above, for the remaining data)

error_path_1 = root + "Errors_siHAU6.xlsx"
error_path_2 = root + "Errors_siLUC1.xlsx"

error_paths = [error_path_1, error_path_2]

def parse_error(error_list = error_paths):
    dfs = []
    for path in error_list:
        df = pd.read_excel(path)
        dfs.append(df)
    return pd.concat(dfs)

df_error = parse_error().replace({"T": True, "F": False, "R": False})

# Fix dtypes in evaluation columns 
# (Mitotic death exluded from this list!)

cols = [
    "Laggings", 
    "Massive Missegregation", 
    "DNA bridge", 
    "Misaligned", 
    "Multipolar Division", 
    'Cytokinesis Failure',
    'Slippage',
    'MN' 
       ]

# Work on a copy to avoid warnings
bool_df = df_error.loc[:, cols].copy()

# Convert and cast to nullable boolean
bool_df = bool_df.apply(lambda col: col.astype("boolean"))
bool_df.dtypes

/var/folders/lr/cb1nd6l97696j8chvkss86580000gn/T/ipykernel_31001/299789669.py:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_error = parse_error().replace({"T": True, "F": False, "R": False})


Laggings                  boolean
Massive Missegregation    boolean
DNA bridge                boolean
Misaligned                boolean
Multipolar Division       boolean
Cytokinesis Failure       boolean
Slippage                  boolean
MN                        boolean
dtype: object

In [ ]:
# Convert values: keep empty as NaN, only "True"/True as True, "False"/False as False
def to_bool_or_nan(x):
    if x in [True, "True"]:
        return True
    elif x in [False, "False"]:
        return False
    else:
        return np.nan   # preserve empties

bool_df.loc[:, "any_true"] = bool_df.any(axis = 1, bool_only = True)

# New column: True if any True, NaN if all are NaN, otherwise False
df_error.loc[:, "any_true"] = bool_df.any(axis = 1)

# cropping V2 errors before concatenating vertically

df_errors_V2 = df_errors_V2[['Source_ID', 'Dataset', 'Position', 'Laggings',
       'Massive Missegregation', 'DNA bridge', 'Misaligned', 'MN',
       'Multipolar Division', 'Cytokinesis Failure', 'Slippage', 'Other',
       'Mitotic Death', 'any_true']]

df_error = pd.concat([df_error, df_errors_V2])

In [10]:
# parse downstream endpoints dataframes after analysis (FIJI)

def getdeaths(filepath, position, dataset):
    deaths = pd.read_csv(filepath, usecols = ["Death", "EndPoint_ID", "Track_ID", "Downstream_IDs"])
    deaths["Position"] = position
    deaths["Dataset"] = dataset 
    return deaths

# Define metadata for each file
deaths_info = [
    {"date": "20250724", "position": 2, "target": "siLUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 3, "target": "siLUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 4, "target": "siLUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 5, "target": "siLUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 6, "target": "siHAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 7, "target": "siHAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 8, "target": "siHAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    {"date": "20250724", "position": 9, "target": "siHAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_downstream_Filled.csv"},
    
    {"date": "20250807", "position": 2, "target": "siLUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 3, "target": "siLUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 4, "target": "siLUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 5, "target": "siHAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 6, "target": "siHAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 7, "target": "siHAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 8, "target": "siHAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},
    {"date": "20250807", "position": 9, "target": "siHAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_downstream_Filled.csv"},

    {"date": "20250814", "position": 2, "target": "siLUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 3, "target": "siLUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 4, "target": "siLUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 5, "target": "siHAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 6, "target": "siHAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 7, "target": "siHAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},
    {"date": "20250814", "position": 8, "target": "siHAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_downstream_Filled.csv"},

    {"date": "20250627", "position": 3, "target": "MonastrolWashout", "file": "20250627_IM_H2B-GFP_MitoStop_02_downstream_Filled.csv"},
    {"date": "20250627", "position": 4, "target": "MonastrolWashout", "file": "20250627_IM_H2B-GFP_MitoStop_02_downstream_Filled.csv"},

    {"date": "20250704", "position": 1, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_downstream_Filled.csv"},
    {"date": "20250704", "position": 2, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_downstream_Filled.csv"},
    {"date": "20250704", "position": 3, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_downstream_Filled.csv"},
    {"date": "20250704", "position": 4, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_downstream_Filled.csv"},
    {"date": "20250704", "position": 5, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_downstream_Filled.csv"},
    {"date": "20250704", "position": 6, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_downstream_Filled.csv"},
    {"date": "20250704", "position": 7, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_downstream_Filled.csv"},
   # {"date": "20250814", "position": 9, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
]

deaths_info2 = [
    {"date": "20251204", "position": "01", "target": "siLUC1", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_downstream_Filled.csv"},
    {"date": "20251204", "position": "02", "target": "siLUC1", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_downstream_Filled.csv"},
    {"date": "20251204", "position": "03", "target": "siLUC1", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_downstream_Filled.csv"},
    {"date": "20251204", "position": "04", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_downstream_Filled.csv"},
    {"date": "20251204", "position": "05", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_downstream_Filled.csv"},
    {"date": "20251204", "position": "06", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_downstream_Filled.csv"},
    {"date": "20251204", "position": "07", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_downstream_Filled.csv"},
    {"date": "20251204", "position": "08", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_downstream_Filled.csv"},
    {"date": "20251204", "position": "09", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_downstream_Filled.csv"},
    {"date": "20251204", "position": "10", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_downstream_Filled.csv"},
    {"date": "20251204", "position": "11", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_downstream_Filled.csv"},
 
    {"date": "20251215", "position": "03", "target": "siLUC1", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_downstream_Filled.csv"},
    {"date": "20251215", "position": "04", "target": "siLUC1", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_downstream_Filled.csv"},
    {"date": "20251215", "position": "05", "target": "siLUC1", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_downstream_Filled.csv"},
    {"date": "20251215", "position": "06", "target": "siLUC1", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_downstream_Filled.csv"},
    {"date": "20251215", "position": "07", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_downstream_Filled.csv"},
    {"date": "20251215", "position": "08", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_downstream_Filled.csv"},
    {"date": "20251215", "position": "09", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_downstream_Filled.csv"},
    {"date": "20251215", "position": "10", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_downstream_Filled.csv"},
    {"date": "20251215", "position": "11", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_downstream_Filled.csv"},
    {"date": "20251215", "position": "12", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_downstream_Filled.csv"},
    {"date": "20251215", "position": "13", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_downstream_Filled.csv"},
    {"date": "20251215", "position": "14", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_downstream_Filled.csv"},
  
    {"date": "20260126_USP28", "position": "01", "target": "siLUC1", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "02", "target": "siLUC1", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "03", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "04", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "05", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "06", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "07", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "08", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "09", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "10", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "11", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "12", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "13", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},
    {"date": "20260126_USP28", "position": "14", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_downstream_Filled.csv"},

    {"date": "20260205_USP28", "position": "01", "target": "siLUC", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "02", "target": "siLUC", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "03", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "04", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "05", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "06", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "07", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "08", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "09", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "10", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "11", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "12", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "13", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260205_USP28", "position": "14", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},

    {"date": "20260306_USP28", "position": "01", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "02", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "03", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "04", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "05", "target": "siLUC1siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "06", "target": "siLUC1siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "07", "target": "siLUC1siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "08", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "09", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "10", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    {"date": "20260306_USP28", "position": "11", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
    
   # {"date": "20260306_USP28", "position": "12", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"}, # empty

  #  {"date": "\20260115_media", "position": "07", "target": "F10+20FBS", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_downstream_Filled.csv"},
]


# Generate and combine dataframes
df_deaths_list = []

for info in deaths_info:
    folder = f"{root}/{info['date']}/Position_{info['position']}_{info['target']}/Concatenated"
    filepath = os.path.join(folder, info["file"])
    df_single = getdeaths(filepath, info["position"], info["date"])
    df_deaths_list.append(df_single)

for info in deaths_info2:
    base_folder = f"{root}/{info['date']}/Position_{info['position']}_{info['target']}"

    folder = None
    for f in os.listdir(base_folder):
        if f.lower() == "max":
            folder = os.path.join(base_folder, f)
            break

    filepath = os.path.join(folder, info["file"])
    df_single = getdeaths(filepath, info["position"], info["date"])
    
    df_deaths_list.append(df_single)

df_deaths_proto = pd.concat(df_deaths_list, ignore_index = True)
df_deaths_proto["EndPoint_ID"] = df_deaths_proto.EndPoint_ID.astype(str)
df_deaths_proto["Death"] = df_deaths_proto.Death.astype(str)
df_deaths_proto["Dataset"] = df_deaths_proto.Dataset.str[:8]

df_deaths = df_deaths_proto[(df_deaths_proto["Death"] == "TRUE")]
df_leaves = df_deaths_proto[(df_deaths_proto["Death"] == "FALSE")]
df_deaths["Death"] = df_deaths["Death"].str.lower().map({"true": True, "false": False})
df_leaves["Leaves"] = df_leaves["Death"].str.lower().map({"true": False, "false": True})

df["EndPoint_ID"] = df["EndPoint_ID"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Collect all endpoint IDs from df (flatten the lists)
all_df_endpoints = set(e for lst in df["EndPoint_ID"] if isinstance(lst, list) for e in lst)

# Collect all death endpoint IDs
all_dead_endpoints = set(df_deaths.loc[df_deaths["Death"] == True, "EndPoint_ID"])

# Collect all Leaves endpoint IDs
all_leaves_endpoints = set(df_leaves.loc[df_leaves["Leaves"] == True, "EndPoint_ID"])

print("Number of endpoints in df:", len(all_df_endpoints))
print("Number of death endpoints:", len(all_dead_endpoints))
print("Number of death overlaps:", len(all_df_endpoints & all_dead_endpoints))
print("Number of leaves endpoints:", len(all_leaves_endpoints))
print("Number of leaves overlaps:", len(all_df_endpoints & all_leaves_endpoints))

(1466, 6)
(613, 6)
(440, 6)
Number of endpoints in df: 1952
Number of death endpoints: 588
Number of death overlaps: 537
Number of leaves endpoints: 426
Number of leaves overlaps: 403


/var/folders/lr/cb1nd6l97696j8chvkss86580000gn/T/ipykernel_31001/597492605.py:168: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_deaths["Death"] = df_deaths["Death"].str.lower().map({"true": True, "false": False})
/var/folders/lr/cb1nd6l97696j8chvkss86580000gn/T/ipykernel_31001/597492605.py:169: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_leaves["Leaves"] = df_leaves["Death"].str.lower().map({"true": False, "false": True})


In [ ]:
dead_map = (
    df_deaths[df_deaths["Death"]]
    .groupby(["Track_ID", "Dataset"])["EndPoint_ID"]
    .apply(set)
)

leaves_map = (
    df_leaves[df_leaves["Leaves"]]
    .groupby(["Track_ID", "Dataset"])["EndPoint_ID"]
    .apply(set)
)

def count_dead_offspring(row):
    dead_endpoints = dead_map.get((row["Track_ID"], row["Dataset"]), set())
    endpoints = row["EndPoint_ID"]

    if isinstance(endpoints, list):
        return sum(e in dead_endpoints for e in endpoints)
    return 0

df["N_dead_offspring"] = df.apply(count_dead_offspring, axis=1)

def count_leaving_offspring(row):
    leaves_endpoints = leaves_map.get((row["Track_ID"], row["Dataset"]), set())
    endpoints = row["EndPoint_ID"]

    if isinstance(endpoints, list):
        return sum(e in leaves_endpoints for e in endpoints)
    return 0

df["N_leaving_offspring"] = df.apply(count_leaving_offspring, axis=1)

df_error["Unique_Source_ID"] = df_error.Dataset.astype(str) + df_error.Position.astype(str) + df_error.Source_ID.astype(str)

# prepare for merging
df["Dataset"] = df["Dataset"].astype(str)
df_error["Dataset"] = df_error["Dataset"].astype(str)

# Final (inner) merge with error dataframe to only include evaluated divisions
df = df.merge(df_error, how = "inner")

In [16]:
def getonsets(filepath, position, dataset):
    onsets = pd.read_csv(filepath)
    onsets["Source_ID"] = onsets.Label.str.split(":").str.get(1).astype(int)
    onsets["Position"] = position
    onsets["Dataset"] = dataset
    onsets.rename(columns = {"Slice": "Frame_onset"}, inplace = True)
    onsets["Frame_onset"] = onsets.Frame_onset - 1 # this corrects for the +1 frame set by FIJI
    return onsets

# Define metadata for each file
onsets_info = [
    {"date": "20250724", "position": 2, "target": "siLUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 3, "target": "siLUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 4, "target": "siLUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 5, "target": "siLUC1", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 6, "target": "siHAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 7, "target": "siHAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 8, "target": "siHAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    {"date": "20250724", "position": 9, "target": "siHAUS6", "file": "20250724_IM_H2B-GFP_MitoStop_RNAi03_onsets.csv"},
    
    {"date": "20250807", "position": 2, "target": "siLUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 3, "target": "siLUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 4, "target": "siLUC1", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 5, "target": "siHAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 6, "target": "siHAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 7, "target": "siHAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 8, "target": "siHAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
    {"date": "20250807", "position": 9, "target": "siHAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},

    {"date": "20250814", "position": 2, "target": "siLUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 3, "target": "siLUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 4, "target": "siLUC1", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 5, "target": "siHAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 6, "target": "siHAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 7, "target": "siHAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},
    {"date": "20250814", "position": 8, "target": "siHAUS6", "file": "20250814_IM_H2B-GFP_MitoStop_RNAi05_onsets.csv"},

    {"date": "20250627", "position": 3, "target": "MonastrolWashout", "file": "20250627_IM_H2B-GFP_MitoStop_02_onsets.csv"},
    {"date": "20250627", "position": 4, "target": "MonastrolWashout", "file": "20250627_IM_H2B-GFP_MitoStop_02_onsets.csv"},

    {"date": "20250704", "position": 1, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 2, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 3, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 4, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 5, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 6, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 7, "target": "MonastrolWashout", "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
   ### {"date": "20250814", "position": 9, "target": "HAUS6", "file": "20250807_IM_H2B-GFP_MitoStop_RNAi04_onsets.csv"},
]

#### Second generation data sets (HAUS6 only) and USP28 double depletion
# Define metadata for each file
onsets_info2 = [
    {"date": "20251204", "position": "01", "target": "siLUC1", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "02", "target": "siLUC1", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "03", "target": "siLUC1", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "04", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "05", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "06", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "07", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "08", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "09", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "10", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    {"date": "20251204", "position": "11", "target": "siHAUS6", "file": "20251204_IM_H2B-GFP_MitoStop_RNAi201_onsets.csv"},
    
    {"date": "20251215", "position": "01", "target": "Mock", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "02", "target": "Mock", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "03", "target": "siLUC1", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "04", "target": "siLUC1", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "05", "target": "siLUC1", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "06", "target": "siLUC1", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "07", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "08", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "09", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "10", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "11", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "12", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "13", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    {"date": "20251215", "position": "14", "target": "siHAUS6", "file": "20251215_IM_H2B-GFP_MitoStop_RNAi202_onsets.csv"},
    
    {"date": "20260126_USP28", "position": "01", "target": "siLUC1", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "02", "target": "siLUC1", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "03", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "04", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "05", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "06", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "07", "target": "siLUC1siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "08", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "09", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "10", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "11", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "12", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "13", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},
    {"date": "20260126_USP28", "position": "14", "target": "siUSP28siHAUS6", "file": "20260126_IM_H2B-GFP_MitoStop_RNAiUSP28_01_onsets.csv"},

    
    {"date": "20260205_USP28", "position": "01", "target": "siLUC", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "02", "target": "siLUC", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "03", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "04", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "05", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "06", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "07", "target": "siLUCsiHAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "08", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "09", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "10", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "11", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "12", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "13", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260205_USP28", "position": "14", "target": "siUSP28HAUS6", "file": "20260205_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},

    {"date": "20260306_USP28", "position": "01", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "02", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "03", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "04", "target": "siLUC1", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "05", "target": "siLUC1siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "06", "target": "siLUC1siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "07", "target": "siLUC1siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "08", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "09", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "10", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "11", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},
    {"date": "20260306_USP28", "position": "12", "target": "siUSP28siHAUS6", "file": "20260306_IM_H2B-GFP_MitoStop_RNAiUSP28_onsets.csv"},

    {"date": "20260115_media", "position": "07", "target": "F10+20FBS", "file": "20260115_IM_H2B-GFP_MitoStop_MediumCtrl01_onsets.csv"},
    {"date": "20260115_media", "position": "08", "target": "F10+20FBS", "file": "20260115_IM_H2B-GFP_MitoStop_MediumCtrl01_onsets.csv"},
    {"date": "20260115_media", "position": "09", "target": "F10+20FBS", "file": "20260115_IM_H2B-GFP_MitoStop_MediumCtrl01_onsets.csv"},
]

# Generate and combine dataframes
df_onsets_list = []

for info in onsets_info:
    folder = f"{root}/{info['date']}/Position_{info['position']}_{info['target']}/Concatenated"
    filepath = os.path.join(folder, info["file"])
    df_single = getonsets(filepath, info["position"], info["date"])
    df_onsets_list.append(df_single)

for info in onsets_info2:
    base_folder = f"{root}/{info['date']}/Position_{info['position']}_{info['target']}"

    folder = None
    for f in os.listdir(base_folder):
        if f.lower() == "max":
            folder = os.path.join(base_folder, f)
            break

    filepath = os.path.join(folder, info["file"])
    df_single = getonsets(filepath, info["position"], info["date"])
    
    df_onsets_list.append(df_single)

df_onsets = pd.concat(df_onsets_list, ignore_index = True)

df_onsets["Position"] = df_onsets.Position.astype(int)
df_onsets["Dataset"] = df_onsets.Dataset.str[:8]
df_onsets["Dataset"] = df_onsets.Dataset.astype(int)
df_onsets["Unique_Source_ID"] = df_onsets.Dataset.astype(str) + df_onsets.Position.astype(str) + df_onsets.Source_ID.astype(str)

In [17]:
# Find duplicate rows
unique_source_ids = df_onsets[["Unique_Source_ID"]]
duplicates = unique_source_ids[unique_source_ids.duplicated()]
print(duplicates)

Empty DataFrame
Columns: [Unique_Source_ID]
Index: []


In [19]:
# Find duplicate rows in main dataframe
unique_source_ids = df[["Unique_Source_ID"]]
duplicates = unique_source_ids[unique_source_ids.duplicated()]
print(duplicates)

     Unique_Source_ID
1290    2025120463037
1291    2025120462755
1292    2025120462708
1293    2025120462038
1294    2025120462177
...               ...
1347    2025120462089
1348    2025120462673
1349    2025120463093
1350    2025120462641
1351    2025120462194

[62 rows x 1 columns]


In [ ]:
# Drop obsolete columns / rows

df = df.drop(columns = ['Unnamed: 0', 'Seen_sister', 'Seen_granny', 'EndPoint_ID'])

df = df.drop_duplicates()
df.shape

In [24]:
df["Frame"] = df.Frame.astype(int) + 1

df_onsets["Dataset"] = df_onsets["Dataset"].astype(str)
df = df.merge(df_onsets, how = "inner") 

(494, 34)
(494, 39)


In [27]:
def getInterphaseDuration(x, dataframe = df):
    # should be the same as cell_cycle - mitotic duration
    daughter_time = x.Frame_onset
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID) 
        ]
        if mother_row["Frame"].shape[0] == 1:   
            mother_time = mother_row["Frame"].item()
            interphase_duration = (int(daughter_time) - int(mother_time)) * 7
            return interphase_duration
        else:
            return None


def getMitoticDurationMother(x, dataframe = df):
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.Dataset == x.Dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        
        if len(mother_row) == 1:
            return mother_row.iloc[0].Mitotic_duration_mins
        else:
            return None

def getMetaphaseArrest(x):
    if x < 30:
        return "000-030 min"
    elif (x < 60) & (x > 30):
        return "030-060"
    elif (x < 90) & (x > 60):
        return "060-090"
    elif (x < 120) & (x > 90):
        return "090-120"
    else:
        return "> 120 min"

def getMetaphaseArrest_mother(x):
    if x > 0:
        if x < 30:
            return "000-030 min"
        elif (x < 60) & (x > 30):
            return "030-060"
        elif (x < 90) & (x > 60):
            return "060-090"
        elif (x < 120) & (x > 90):
            return "090-120"
        else:
            return "> 120 min"
    else:
        pass

df["Frames_in_mitosis"] = (df.Frame - df.Frame_onset.astype(int))
df["Mitotic_duration_mins"] = df.Frames_in_mitosis * frame_interval

df["Interphase_duration_mins"] = df.apply(getInterphaseDuration, axis = 1) 
df["Interphase_duration_hrs"] = df.Interphase_duration_mins / 60
df["Mitotic_duration_hrs"] = df.Mitotic_duration_mins / 60

df["Mother_Mitotic_duration_mins"] = df.apply(getMitoticDurationMother, axis = 1)
df["Mother_arrested"] = df.Mother_Mitotic_duration_mins.apply(getMetaphaseArrest_mother)
df["Metaphase_arrested"] = df.Mitotic_duration_mins.apply(getMetaphaseArrest)
df["Frame_onset_hrs"] = df.Frame_onset * frame_interval / 60 # Augmin depletion time at NEB
df["Frame_hrs"] = df.Frame * frame_interval / 60 # Augmin depletion time at anaphase

# Create Bins (one bin for every 12 h of depletion, binning cells with the same onset of mitotis)
interval_range_depletion = pd.interval_range(start = 12, freq = 24, end = 84) 
df['Augmin_depletion_bin'] = pd.cut(df['Frame_onset_hrs'], bins = interval_range_depletion).astype(str)

# Create Bins (one bin for every 30 min of mitotic duration) # kind of the same as "Metaphase_arrested"
interval_range_mitosis = pd.interval_range(start = 0, freq = 30, end = 900) 
df['Mitotic_duration_bin'] = pd.cut(df['Mitotic_duration_mins'], bins = interval_range_mitosis).astype(str)

df["Unique_Track_ID"] = df.Dataset.astype(str) + df.Position.astype(str) + df.Track_ID.astype(str)
df["Unique_Source_ID"] = df.Dataset.astype(str) + df.Position.astype(str) + df.Source_ID.astype(str)

# Drop annotations with mistakes
df = df.drop(df[df.Cell_Cycle_hours < 1].index)
df = df.drop(df[df.Mitotic_duration_mins < 8].index)
df = df.drop(df[df.Mother_Mitotic_duration_mins < 8].index)

,Track_ID,Source_ID,Splitting_event,Lineage,Track_Coordinate_X,Track_Coordinate_Y,Frame,Position,Dataset,Condition,...,Interphase_duration_hrs,Mitotic_duration_hrs,Mother_Mitotic_duration_mins,Mother_arrested,Metaphase_arrested,Frame_onset_hrs,Frame_hrs,Augmin_depletion_bin,Mitotic_duration_bin,Unique_Track_ID
0,3,2153,True,2153,874.632911,867.175122,4,2,20250724,siLUC1,...,NaN,0.466667,NaN,None,000-030 min,0.000000,0.466667,nan,"(0, 30]",2025072423
1,14,2989,True,2989,1014.949828,559.472276,105,2,20250724,siLUC1,...,NaN,0.933333,NaN,None,030-060,11.316667,12.250000,nan,"(30, 60]",20250724214
2,15,3052,True,3052,1051.410130,420.260216,111,2,20250724,siLUC1,...,NaN,0.816667,NaN,None,030-060,12.133333,12.950000,"(12, 36]","(30, 60]",20250724215
3,0,2005,True,2005,666.091036,713.323699,118,2,20250724,siLUC1,...,NaN,1.166667,NaN,None,060-090,12.600000,13.766667,"(12, 36]","(60, 90]",2025072420
4,2,2121,True,2121,499.809964,637.640952,119,2,20250724,siLUC1,...,NaN,1.400000,NaN,None,060-090,12.483333,13.883333,"(12, 36]","(60, 90]",2025072422


In [31]:
# Correct daughter deaths with mitotic deaths

def correct_death(x):
    if x["Mitotic Death"] == True:
        return x.N_dead_offspring - 2
    else:
        return x.N_dead_offspring

def death_bool(x):
    if x > 0:
        return True
    else:
        return False

df["N_dead_offspring"] = df.apply(correct_death, axis = 1) 
df["N_dead_offspring_bool"] = df.N_dead_offspring.apply(death_bool)

In [32]:
def conditions(x):
    if x == "siHAUS6":
        return "2_siHAUS6"
    if x == "siLUC1siHAUS6":
        return "3_siLUC1siHAUS6"
    if x == "siUSP28siHAUS6":
        return "4_siUSP28siHAUS6"
    if x == "siMock":
        return "0_siMock"
    if x == "F1020FBS":
        return "F10"
    if x == "Monastrol":
        return "Monastrol"
    else:
        return "1_siLUC1"

df["Condition"] = df.Condition.apply(conditions)

In [33]:
df.to_csv(outputFolder + "MainDataFrame_MitoticStopwatch_Augmin.csv")

print("Finished.")

Finished.
